# Lesson 5A: N-Step Methods & Eligibility Traces - Theory

## Introduction

So far we've learned:
- **DP**: Uses full model (not practical)
- **MC**: Uses full returns (high variance)
- **TD(0)**: Uses 1-step bootstrap (low variance but biased)

**N-step methods** interpolate: bootstrap after n steps instead of 1.

## Key Insight

By varying n, we slide between TD and MC:
- n=1: TD(0) (immediate bootstrap)
- n=∞: MC (full trajectory)
- n∈(1,∞): Balanced bias-variance

**Eligibility traces** make this efficient by tracking which states/actions contributed to prediction errors.

## Setup & Core Ideas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print('N-Step TD and Eligibility Traces')

## N-Step Returns

### Definition

**n-step return**:
$$G_t^{(n)} = R_{t+1} + \gamma R_{t+2} + ... + \gamma^{n-1} R_{t+n} + \gamma^n V(S_{t+n})$$

Sum first n rewards, then bootstrap off state at t+n.

### Special Cases

- **n=1**: $G_t^{(1)} = R_{t+1} + \gamma V(S_{t+1})$ (TD)
- **n=∞**: $G_t^{(∞)} = R_{t+1} + \gamma R_{t+2} + ...$ (MC)

### Bias-Variance

- n↑: More actual rewards → lower bias, higher variance
- n↓: More bootstrapping → higher bias, lower variance

## N-Step TD Prediction

### Update Rule

$$V(S_t) \leftarrow V(S_t) + \alpha[G_t^{(n)} - V(S_t)]$$

Waits n steps before updating. Simple but inefficient.

### Disadvantage

Can only update every n steps. Slow in practice.

## Eligibility Traces

### Core Idea

Instead of waiting n steps, **update all visited states immediately** with eligibility weights.

### Forward View

$$e_t(s) = \gamma \lambda e_{t-1}(s) + \mathbb{1}_{s=S_t}$$

- Decay past eligibilities by γλ
- Add 1 when state s is visited

### TD(λ) Update

$$V(s) \leftarrow V(s) + \alpha \delta_t e_t(s)$$ for all s

where $\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$ (TD error)

### Interpretation

- Each visited state gets credit weighted by recency
- λ controls credit decay:
  - λ=0: Only current state credited (TD(0))
  - λ=1: Full credit to all visited states (MC)
  - λ∈(0,1): Geometrically decayed credit

## Forward-Backward View Equivalence

### Forward View

$$V(S_t) \leftarrow V(S_t) + \alpha[G_t^{\lambda} - V(S_t)]$$

where $G_t^{\lambda} = (1-\lambda)\sum_{n=1}^{\infty} \lambda^{n-1} G_t^{(n)}$

Weighted average of all n-step returns.

### Backward View

$$\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$$
$$e_t(s) = \gamma \lambda e_{t-1}(s) + \mathbb{1}_{s=S_t}$$
$$V(s) \leftarrow V(s) + \alpha \delta_t e_t(s)$$ for all s

### The Key Theorem

**Forward and backward views are mathematically equivalent.**

Choosing eligibility traces is just a computational trick to implement n-step returns efficiently.

## Sarsa(λ): On-Policy TD(λ) Control

### Algorithm

```
Initialize e(s,a) = 0 for all s,a

loop:
  S, A ← state, action from ε-greedy
  S', R ← environment step
  
  δ ← R + γQ(S',A') - Q(S,A)
  e(S,A) ← e(S,A) + 1  # accumulating trace
  
  for all s,a:
    Q(s,a) ← Q(s,a) + α δ e(s,a)
    e(s,a) ← γλ e(s,a)  # decay traces
  
  S,A ← S',A'
```

Updates Q immediately with eligibility weighting.

## Summary

### Key Concepts

1. **N-step returns**: Blend n rewards with bootstrap
2. **Eligibility traces**: Track state recency for credit assignment
3. **TD(λ)**: Weighted average of n-step returns via traces
4. **Forward/backward equivalence**: Different view, same algorithm
5. **Sarsa(λ)**: On-policy control with eligibility traces

### Why This Matters

- Interpolates between TD and MC smoothly
- Efficient credit assignment to multiple states per step
- Faster convergence than TD(0) in many domains
- Foundation for modern deep RL (eligibility traces in experience replay)

### What We Implemented

- **RandomWalk environment**: Simple 1D gridworld (states 0–9) using Gymnasium API
- **Sarsa(λ) from scratch**: Backward-view eligibility traces with accumulating trace
- **Learning comparison**: λ=0.9 vs TD(0)—showed how eligibility traces accelerate credit assignment

### Key Insights

- λ=0 (TD): Only updates current state, slower credit propagation
- λ=0.9: Traces decay eligibly, spreading credit to recent states proportionally
- Trade-off: Higher λ → more distributed credit, but higher variance; lower λ → lower variance, but slower learning

### Next: Lesson 5B (Practical)

Expand to Stable-Baselines3 reference implementation and CartPole continuous domain.

In [ ]:
plt.figure(figsize=(10, 5))

# Smooth the returns for cleaner plot
window = 5
returns_td_smooth = np.convolve(returns_td, np.ones(window)/window, mode='valid')
returns_lambda_smooth = np.convolve(returns_lambda, np.ones(window)/window, mode='valid')

plt.plot(returns_td_smooth, label='λ=0.0 (TD)', alpha=0.7, linewidth=2)
plt.plot(returns_lambda_smooth, label='λ=0.9 (Eligibility Traces)', alpha=0.7, linewidth=2)
plt.xlabel('Episode')
plt.ylabel('Episode Return')
plt.title('Sarsa(λ): Effect of λ on Learning Speed')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"TD(0) final return: {returns_td[-1]:.1f}")
print(f"Sarsa(λ=0.9) final return: {returns_lambda[-1]:.1f}")
if returns_td[-1] != 0:
    improvement = (returns_lambda[-1] - returns_td[-1]) / abs(returns_td[-1]) * 100
    print(f"Improvement: {improvement:.1f}%")

### Results: λ Effect on Learning

Plot the learning curves to show how λ affects convergence.

In [ ]:
def train_sarsa_lambda(env, agent, num_episodes=100, max_steps=100, lambda_val=0.9):
    """Train Sarsa(λ) and track returns"""
    returns = []

    for episode in range(num_episodes):
        agent.lambda_ = lambda_val
        agent.reset_traces()
        state, _ = env.reset()
        action = agent.get_action(state)

        episode_return = 0
        for step in range(max_steps):
            next_state, reward, done, _, _ = env.step(action)
            next_action = agent.get_action(next_state)

            agent.update(state, action, reward, next_state, next_action, done)

            episode_return += reward
            state, action = next_state, next_action

            if done:
                break

        returns.append(episode_return)

    return returns

# Train with different λ values
print("Training Sarsa(λ) with λ=0.0 (TD(0))...")
env_td = RandomWalk(num_states=10)
agent_td = SarsaLambda(num_states=10, num_actions=2, alpha=0.1, gamma=0.99, lambda_=0.0)
returns_td = train_sarsa_lambda(env_td, agent_td, num_episodes=100, lambda_val=0.0)

print("Training Sarsa(λ) with λ=0.9...")
env_lambda = RandomWalk(num_states=10)
agent_lambda = SarsaLambda(num_states=10, num_actions=2, alpha=0.1, gamma=0.99, lambda_=0.9)
returns_lambda = train_sarsa_lambda(env_lambda, agent_lambda, num_episodes=100, lambda_val=0.9)

print("Training complete!")

### Training Loop

In [ ]:
class SarsaLambda:
    """Sarsa(λ) agent with eligibility traces (backward view)"""

    def __init__(self, num_states, num_actions, alpha=0.1, gamma=0.9, lambda_=0.9, epsilon=0.1):
        self.num_states = num_states
        self.num_actions = num_actions
        self.alpha = alpha
        self.gamma = gamma
        self.lambda_ = lambda_
        self.epsilon = epsilon

        # Q-table and eligibility traces
        self.Q = np.zeros((num_states, num_actions))
        self.e = np.zeros((num_states, num_actions))

    def get_action(self, state):
        """ε-greedy action selection"""
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.num_actions)
        else:
            return np.argmax(self.Q[state])

    def update(self, state, action, reward, next_state, next_action, done):
        """Backward view: TD error + eligibility trace updates"""
        # TD error
        if done:
            delta = reward - self.Q[state, action]
        else:
            delta = reward + self.gamma * self.Q[next_state, next_action] - self.Q[state, action]

        # Accumulating trace for current s,a
        self.e[state, action] += 1

        # Update all Q-values weighted by eligibility
        for s in range(self.num_states):
            for a in range(self.num_actions):
                self.Q[s, a] += self.alpha * delta * self.e[s, a]
                self.e[s, a] *= self.gamma * self.lambda_

    def reset_traces(self):
        """Reset eligibility traces at episode start"""
        self.e.fill(0)

print("SarsaLambda class defined.")

### Sarsa(λ) Implementation (From Scratch)

In [ ]:
import gymnasium as gym
import numpy as np
from gymnasium import spaces

class RandomWalk(gym.Env):
    """1D random walk: states 0-9, start at 5, goal is 9"""

    def __init__(self, num_states=10):
        self.num_states = num_states
        self.start_state = num_states // 2
        self.state = self.start_state
        self.action_space = spaces.Discrete(2)  # 0: left, 1: right
        self.observation_space = spaces.Discrete(num_states)

    def reset(self, seed=None):
        super().reset(seed=seed)
        self.state = self.start_state
        return self.state, {}

    def step(self, action):
        assert self.action_space.contains(action)

        # Action 0: move left, Action 1: move right
        if action == 0:
            self.state -= 1
        else:
            self.state += 1

        # Reward: -1 at left boundary, +1 at right boundary (goal), 0 otherwise
        done = False
        if self.state <= 0:
            reward = -1
            done = True
        elif self.state >= self.num_states - 1:
            reward = 1
            done = True
        else:
            reward = 0

        return self.state, reward, done, False, {}

    def render(self):
        print("State: {}".format(self.state))

# Test the environment
env = RandomWalk(num_states=10)
state, _ = env.reset()
print("RandomWalk initialized. Start state:", state)
print("Action space:", env.action_space)
print("Observation space:", env.observation_space)

## Practical: Sarsa(λ) on RandomWalk

### RandomWalk Environment

A simple 1D environment: start at state 5 (middle of 0-9), +1 for moving right to goal, -1 for moving left to boundary.